# TABA — FEMTO/PRONOSTIA RUL Prognostics with Green AI Trade-off Analysis

**Status: ingestion fix in progress.**

The previous run of this pipeline was invalid: the CSV loader picked up every `*.csv` file under `Learning_set/Bearing*/`, including `temp_*.csv` (single-column temperature logs). This caused a crash when a temperature file reached the vibration reader (`ValueError: Arquivo com menos de duas colunas`), and — worse — the feature table computed before that fix (`features_reference.csv`) contains mostly constant/duplicated rows across the horizontal and vertical channels. That degeneracy explains the suspicious old results (MAE ≈ 38.84 min for Ridge/HGB/RF, R² ≈ -0.39, critical recall = 0, and a diverging MLP).

This notebook rebuilds the pipeline with:
1. Strict `acc_*.csv` selection (vibration only).
2. A mandatory raw-file inspection step before any feature extraction.
3. A hard validation gate after feature extraction that **stops execution** if features are degenerate (too few unique values, too many zeros/NaNs, or h/v channels identical).
4. Model training (Ridge, HistGradientBoosting, RandomForest, MLP) across `reference`, `half-rate`, and `top-10-features` strategies, only once the gate passes.
5. Green AI accounting (train/inference time, energy, CO2e, model size) and a MAE-vs-energy Pareto view.

**Do not skip the validation cells.** If the gate fails, fix the reader/selector before continuing — do not comment out the check.

## 1. Setup

In [ ]:
import io
import pathlib
import shutil
import time
import zipfile

import numpy as np
import pandas as pd
from scipy import stats
from scipy.fft import rfft, rfftfreq

import matplotlib.pyplot as plt

from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import GroupShuffleSplit
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 2. Upload and extract dataset (corrected: acceleration files only)

In [ ]:
from google.colab import files

# Faz upload do ZIP do dataset FEMTO/PRONOSTIA
uploaded = files.upload()

zip_name = next(
    (name for name in uploaded if name.lower().endswith(".zip")),
    None,
)

if zip_name is None:
    raise ValueError("Faça upload de um único arquivo .zip do dataset FEMTO.")

# Limpa a versão anterior para não misturar arquivos de execuções passadas
DATA_DIR = pathlib.Path("/content/femto")

if DATA_DIR.exists():
    shutil.rmtree(DATA_DIR)

DATA_DIR.mkdir(parents=True, exist_ok=True)

# Extrai o ZIP
with zipfile.ZipFile(io.BytesIO(uploaded[zip_name])) as z:
    z.extractall(DATA_DIR)

# IMPORTANTE: usa somente sinais de aceleração/vibração.
# Os arquivos temp_*.csv têm uma coluna e não devem entrar na extração de features.
csv_files = sorted(DATA_DIR.rglob("acc_*.csv"))

print("Acceleration CSV files found:", len(csv_files))

for csv_file in csv_files[:10]:
    print(csv_file)

if len(csv_files) == 0:
    raise ValueError(
        "Nenhum arquivo acc_*.csv foi encontrado. "
        "Confirme se o ZIP contém as pastas Learning_set/Bearing*/."
    )

## 3. Configuration

The 2012 PRONOSTIA data use 25.6 kHz vibration recordings of 0.1 s, collected approximately every 10 s. Confirm any dataset-specific deviation (e.g. FEMTO Challenge repacking) against the audit below before trusting these defaults.

In [ ]:
FS_NATIVE = 25600          # Hz, native sampling rate of PRONOSTIA acc_*.csv files
SAMPLE_INTERVAL_S = 10      # s, nominal interval between consecutive recordings
RUL_CAP_MIN = None          # set to a number of minutes to cap RUL, or None for no cap

FREQ_BANDS_HZ = [(0, 1000), (1000, 5000), (5000, FS_NATIVE // 2)]

## 4. Mandatory raw-file inspection

Run this before writing/trusting any feature-extraction code. Confirm: two numeric columns, the correct separator, and no header row. Adjust the reader in Section 5 if this does not hold.

In [ ]:
sample_path = csv_files[0]
print("Sample file:", sample_path)

with open(sample_path, "r") as f:
    for _ in range(5):
        print(repr(f.readline()))

sample = pd.read_csv(sample_path, header=None)
display(sample.head())
print("Shape:", sample.shape)
print("Dtypes:", sample.dtypes.tolist())

Common PRONOSTIA layouts export 6 columns per row: `hour, minute, second, µsecond, h_accel, v_accel` — vibration lives in the **last two columns**, not the first two. The reader below auto-detects the case and always takes the last two numeric columns as (horizontal, vertical). If the printed shape/head above does not look like that, stop and adjust `read_measurement` before proceeding.

## 5. Feature extraction

In [ ]:
def read_measurement(path: pathlib.Path) -> tuple[np.ndarray, np.ndarray]:
    """Returns (horizontal, vertical) vibration arrays from an acc_*.csv file."""
    df = pd.read_csv(path, header=None, sep=None, engine="python")
    df = df.apply(pd.to_numeric, errors="coerce")
    df = df.dropna(axis=1, how="all")

    if df.shape[1] < 2:
        raise ValueError(f"Arquivo com menos de duas colunas: {path}")

    # Vibration channels are the last two numeric columns regardless of
    # whether timestamp columns (hour/min/sec/usec) precede them.
    h = df.iloc[:, -2].to_numpy(dtype=float)
    v = df.iloc[:, -1].to_numpy(dtype=float)
    return h, v


def band_energy_ratios(signal: np.ndarray, fs: int, bands) -> list[float]:
    n = len(signal)
    spectrum = np.abs(rfft(signal))
    freqs = rfftfreq(n, d=1.0 / fs)
    power = spectrum ** 2
    total = power.sum()
    if total <= 0:
        return [0.0 for _ in bands]
    ratios = []
    for lo, hi in bands:
        mask = (freqs >= lo) & (freqs < hi)
        ratios.append(float(power[mask].sum() / total))
    return ratios


def extract_channel_features(signal: np.ndarray, fs: int) -> list[float]:
    signal = signal.astype(float)
    rms = float(np.sqrt(np.mean(signal ** 2)))
    std = float(np.std(signal))
    p2p = float(np.ptp(signal))
    crest = float(np.max(np.abs(signal)) / rms) if rms > 0 else 0.0
    skew = float(stats.skew(signal))
    kurt = float(stats.kurtosis(signal))
    energy = float(np.sum(signal ** 2))

    spectrum = np.abs(rfft(signal))
    freqs = rfftfreq(len(signal), d=1.0 / fs)
    power = spectrum ** 2
    total_power = power.sum()
    if total_power > 0:
        centroid = float(np.sum(freqs * power) / total_power)
        p_norm = power / total_power
        p_norm = p_norm[p_norm > 0]
        spec_entropy = float(-np.sum(p_norm * np.log2(p_norm)) / np.log2(len(p_norm)))
    else:
        centroid = 0.0
        spec_entropy = 0.0

    band_ratios = band_energy_ratios(signal, fs, FREQ_BANDS_HZ)

    return [rms, std, p2p, crest, skew, kurt, energy, centroid, spec_entropy] + band_ratios


FEATURE_NAMES = [
    "rms", "std", "p2p", "crest", "skew", "kurtosis", "energy",
    "spec_centroid", "spec_entropy", "band_low", "band_mid", "band_high",
]
assert len(FEATURE_NAMES) == 12

In [ ]:
def build_feature_table(csv_files: list[pathlib.Path], fs: int) -> pd.DataFrame:
    by_bearing: dict[str, list[pathlib.Path]] = {}
    for path in csv_files:
        bearing = path.parent.name
        by_bearing.setdefault(bearing, []).append(path)

    for bearing in by_bearing:
        by_bearing[bearing] = sorted(by_bearing[bearing])

    rows = []
    for bearing, paths in by_bearing.items():
        n_obs = len(paths)
        for i, path in enumerate(paths):
            h, v = read_measurement(path)
            h_feats = extract_channel_features(h, fs)
            v_feats = extract_channel_features(v, fs)

            rul_min = (n_obs - 1 - i) * SAMPLE_INTERVAL_S / 60
            if RUL_CAP_MIN is not None:
                rul_min = min(rul_min, RUL_CAP_MIN)

            row = {
                "bearing": bearing,
                "obs_index": i,
                "n_obs_bearing": n_obs,
                "rul_min": rul_min,
            }
            row.update({f"h_{name}": val for name, val in zip(FEATURE_NAMES, h_feats)})
            row.update({f"v_{name}": val for name, val in zip(FEATURE_NAMES, v_feats)})
            rows.append(row)

    return pd.DataFrame(rows)


t0 = time.time()
features = build_feature_table(csv_files, FS_NATIVE)
print(f"Feature extraction took {time.time() - t0:.1f}s")

feature_columns = [c for c in features.columns if c.startswith("h_") or c.startswith("v_")]

print("Rows:", len(features))
print("Bearings:", features["bearing"].nunique())
print("Feature columns:", len(feature_columns))
print("Total NaNs in features:", features[feature_columns].isna().sum().sum())

## 6. Validation gate — do not proceed past this cell if it raises

This is the check that the previous run skipped. It fails loudly instead of silently producing degenerate features.

In [ ]:
print("Número de arquivos de aceleração:", len(csv_files))
print("Formato de uma amostra:", pd.read_csv(csv_files[0], header=None).shape)
print("\nLinhas na tabela:", len(features))
print("Rolamentos:", features["bearing"].nunique())
print("Número de features:", len(feature_columns))
print("NaNs totais:", features[feature_columns].isna().sum().sum())

print("\nNúmero de valores únicos por feature:")
nunique = features[feature_columns].nunique().sort_values()
display(nunique)

print("\nEstatísticas das features:")
display(features[feature_columns].describe().T)

errors = []

if len(csv_files) == 0:
    errors.append("Nenhum arquivo de aceleração encontrado.")

if len(feature_columns) != 24:
    errors.append(f"Esperado 24 colunas de features (12 por canal), encontrado {len(feature_columns)}.")

if features[feature_columns].isna().sum().sum() > 0:
    errors.append("Existem NaNs na tabela de features.")

n_rows = len(features)
near_constant = nunique[nunique <= max(1, int(0.01 * n_rows))]
if len(near_constant) > 0:
    errors.append(
        f"{len(near_constant)} features quase constantes (<=1% de valores únicos): "
        f"{list(near_constant.index)}"
    )

zero_fraction = (features[feature_columns] == 0).mean()
mostly_zero = zero_fraction[zero_fraction > 0.5]
if len(mostly_zero) > 0:
    errors.append(f"Features majoritariamente zeradas (>50%): {list(mostly_zero.index)}")

h_cols = [c for c in feature_columns if c.startswith("h_")]
v_cols = [c.replace("h_", "v_", 1) for c in h_cols]
identical_frac = (features[h_cols].to_numpy() == features[v_cols].to_numpy()).mean()
if identical_frac > 0.5:
    errors.append(
        f"Canais h_* e v_* idênticos em {identical_frac:.0%} das células — "
        "provável erro de leitura de colunas."
    )

if errors:
    raise ValueError(
        "VALIDATION GATE FAILED — corrija a ingestão antes de treinar modelos:\n- "
        + "\n- ".join(errors)
    )

print("\n✅ Validation gate passed. Safe to proceed to training.")

## 7. Train/test split and model training

Split is grouped by `bearing` so no bearing's observations leak across train/test — a run-to-failure sequence is a single unit.

In [ ]:
TARGET = "rul_min"
CRITICAL_RUL_MIN = 20  # observations at/below this RUL are the "critical" class for recall

splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_STATE)
train_idx, test_idx = next(splitter.split(features, groups=features["bearing"]))

train_df = features.iloc[train_idx].reset_index(drop=True)
test_df = features.iloc[test_idx].reset_index(drop=True)

print("Train bearings:", sorted(train_df["bearing"].unique()))
print("Test bearings:", sorted(test_df["bearing"].unique()))

In [ ]:
def top_k_features_by_importance(X_train, y_train, columns, k=10):
    rf = RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1)
    rf.fit(X_train, y_train)
    order = np.argsort(rf.feature_importances_)[::-1]
    return [columns[i] for i in order[:k]]


def get_model(name: str):
    if name == "Ridge":
        return Ridge(alpha=1.0, random_state=RANDOM_STATE)
    if name == "HistGradientBoosting":
        return HistGradientBoostingRegressor(random_state=RANDOM_STATE)
    if name == "RandomForest":
        return RandomForestRegressor(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1)
    if name == "MLP":
        return MLPRegressor(
            hidden_layer_sizes=(64, 32),
            max_iter=2000,
            early_stopping=True,
            random_state=RANDOM_STATE,
        )
    raise ValueError(f"Unknown model: {name}")


MODEL_NAMES = ["Ridge", "HistGradientBoosting", "RandomForest", "MLP"]

In [ ]:
import pickle

try:
    from codecarbon import EmissionsTracker
    HAS_CODECARBON = True
except ImportError:
    HAS_CODECARBON = False
    print("codecarbon not installed — energy/CO2 will be estimated from CPU-time only. "
          "Run `!pip install codecarbon` for measured energy.")

# Rough estimate used only when codecarbon is unavailable (typical laptop/Colab CPU TDP share).
ESTIMATED_WATTS = 15.0
GRID_CARBON_INTENSITY_G_PER_KWH = 429.0  # global average, used only as an estimate fallback


def measure(fn):
    if HAS_CODECARBON:
        tracker = EmissionsTracker(log_level="error", save_to_file=False)
        tracker.start()
        t0 = time.time()
        result = fn()
        elapsed = time.time() - t0
        emissions_kg = tracker.stop()
        energy_kwh = tracker._total_energy.kWh if hasattr(tracker, "_total_energy") else None
        co2_g = (emissions_kg or 0.0) * 1000
    else:
        t0 = time.time()
        result = fn()
        elapsed = time.time() - t0
        energy_kwh = (ESTIMATED_WATTS * elapsed / 3600) / 1000
        co2_g = energy_kwh * GRID_CARBON_INTENSITY_G_PER_KWH
    return result, elapsed, energy_kwh, co2_g


def build_strategy_data(strategy: str, train_df: pd.DataFrame, test_df: pd.DataFrame, all_feature_cols: list[str]):
    """Returns (X_train, X_test, cols_used, downsample_factor) for a given strategy."""
    if strategy == "reference":
        cols = all_feature_cols
        return train_df[cols].to_numpy(), test_df[cols].to_numpy(), cols, 1

    if strategy == "half-rate":
        # Recompute features from downsampled raw signal (fs/2) instead of the
        # native-rate table, so the cost reduction is real, not simulated.
        def rebuild(df):
            rows = []
            for bearing, group in df.groupby("bearing"):
                paths = sorted((DATA_DIR).rglob(f"acc_*.csv"))
                paths = [p for p in paths if p.parent.name == bearing]
                for i in group["obs_index"]:
                    h, v = read_measurement(paths[i])
                    h_ds, v_ds = h[::2], v[::2]
                    fs_ds = FS_NATIVE // 2
                    h_feats = extract_channel_features(h_ds, fs_ds)
                    v_feats = extract_channel_features(v_ds, fs_ds)
                    row = {f"h_{n}": val for n, val in zip(FEATURE_NAMES, h_feats)}
                    row.update({f"v_{n}": val for n, val in zip(FEATURE_NAMES, v_feats)})
                    rows.append(row)
            return pd.DataFrame(rows)[all_feature_cols]

        X_train = rebuild(train_df).to_numpy()
        X_test = rebuild(test_df).to_numpy()
        return X_train, X_test, all_feature_cols, 2

    if strategy == "top-10-features":
        scaler_tmp = StandardScaler().fit(train_df[all_feature_cols])
        top_cols = top_k_features_by_importance(
            scaler_tmp.transform(train_df[all_feature_cols]),
            train_df[TARGET].to_numpy(),
            all_feature_cols,
            k=10,
        )
        return train_df[top_cols].to_numpy(), test_df[top_cols].to_numpy(), top_cols, 1

    raise ValueError(f"Unknown strategy: {strategy}")

In [ ]:
STRATEGIES = ["reference", "half-rate", "top-10-features"]

results = []

for strategy in STRATEGIES:
    print(f"\n=== Strategy: {strategy} ===")
    X_train_raw, X_test_raw, cols_used, downsample = build_strategy_data(
        strategy, train_df, test_df, feature_columns
    )

    scaler = StandardScaler().fit(X_train_raw)
    X_train = scaler.transform(X_train_raw)
    X_test = scaler.transform(X_test_raw)
    y_train = train_df[TARGET].to_numpy()
    y_test = test_df[TARGET].to_numpy()

    for model_name in MODEL_NAMES:
        model = get_model(model_name)

        _, train_time, train_energy_kwh, train_co2_g = measure(lambda: model.fit(X_train, y_train))
        y_pred, infer_time, infer_energy_kwh, infer_co2_g = measure(lambda: model.predict(X_test))

        mae = mean_absolute_error(y_test, y_pred)
        rmse = mean_squared_error(y_test, y_pred, squared=False)
        r2 = r2_score(y_test, y_pred)

        critical_mask = y_test <= CRITICAL_RUL_MIN
        if critical_mask.sum() > 0:
            pred_flagged = y_pred <= CRITICAL_RUL_MIN
            critical_recall = float((pred_flagged & critical_mask).sum() / critical_mask.sum())
        else:
            critical_recall = float("nan")

        model_size_kb = len(pickle.dumps(model)) / 1024

        results.append({
            "strategy": strategy,
            "model": model_name,
            "n_features": len(cols_used),
            "downsample_factor": downsample,
            "mae_min": mae,
            "rmse_min": rmse,
            "r2": r2,
            "critical_recall": critical_recall,
            "train_time_s": train_time,
            "infer_time_s": infer_time,
            "train_energy_kwh": train_energy_kwh,
            "co2_g": train_co2_g + infer_co2_g,
            "model_size_kb": model_size_kb,
        })
        print(f"  {model_name:22s} MAE={mae:8.2f} min  R2={r2:6.3f}  "
              f"recall_crit={critical_recall:.2f}  train_t={train_time:.2f}s")

results_df = pd.DataFrame(results)
display(results_df)

## 8. Pareto view: MAE vs. energy

In [ ]:
def is_pareto_efficient(costs: np.ndarray) -> np.ndarray:
    """costs: (n, 2) array, lower is better on both axes."""
    is_efficient = np.ones(costs.shape[0], dtype=bool)
    for i, c in enumerate(costs):
        if is_efficient[i]:
            is_efficient[is_efficient] = np.any(costs[is_efficient] < c, axis=1) | np.all(costs[is_efficient] == c, axis=1)
            is_efficient[i] = True
    return is_efficient


pareto_costs = results_df[["mae_min", "train_energy_kwh"]].to_numpy()
results_df["pareto_optimal"] = is_pareto_efficient(pareto_costs)

fig, ax = plt.subplots(figsize=(8, 6))
for strategy in STRATEGIES:
    sub = results_df[results_df["strategy"] == strategy]
    ax.scatter(sub["train_energy_kwh"], sub["mae_min"], label=strategy, s=80)
    for _, row in sub.iterrows():
        ax.annotate(row["model"], (row["train_energy_kwh"], row["mae_min"]), fontsize=8)

pareto_pts = results_df[results_df["pareto_optimal"]]
ax.scatter(pareto_pts["train_energy_kwh"], pareto_pts["mae_min"],
           facecolors="none", edgecolors="black", s=200, linewidths=1.5, label="Pareto-optimal")

ax.set_xlabel("Training energy (kWh)")
ax.set_ylabel("MAE (min)")
ax.set_title("MAE vs. training energy — FEMTO/PRONOSTIA RUL")
ax.legend()
plt.tight_layout()
plt.savefig("pareto_mae_energy.jpeg", dpi=150)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
pivot = results_df.pivot(index="model", columns="strategy", values="mae_min")
pivot.plot(kind="bar", ax=ax)
ax.set_ylabel("MAE (min)")
ax.set_title("MAE by model and strategy")
plt.tight_layout()
plt.savefig("mae_models.jpeg", dpi=150)
plt.show()

## 9. Export

In [ ]:
OUT_DIR = pathlib.Path("/content/taba_femto_outputs")
OUT_DIR.mkdir(exist_ok=True)

features.to_csv(OUT_DIR / "features_reference.csv", index=False)
results_df[results_df["strategy"] == "reference"].to_csv(OUT_DIR / "reference_results.csv", index=False)
results_df[results_df["strategy"] != "reference"].to_csv(OUT_DIR / "green_strategy_results.csv", index=False)
results_df.to_csv(OUT_DIR / "all_results_pareto.csv", index=False)

print("Saved to", OUT_DIR)
for p in sorted(OUT_DIR.iterdir()):
    print(" ", p.name)